# Analysis segment effect matrix v1

目标：把 `src/analysis` 的 qlib/regime/window 分析再压成可 review 的分段模型差异矩阵和中文 briefing。

外部参考：Qlib 的 `PortAnaRecord`/recorder 产出 backtest report、positions、risk analysis；Qlib report 模块用于可视化 portfolio 与 model performance。本轮不把 qlib 图形逻辑塞进 notebook，而是把 recorder/report 思路落到稳定 CSV/manifest 表。

In [1]:
from pathlib import Path
import sys
import pandas as pd

REPO_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'AGENTS.md').exists() and (candidate / 'src').exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis import (
    analyze_model_effects_by_regime_and_window,
    build_model_review_briefing,
    build_regime_window_analysis_report,
    compare_models,
    compute_regime_metrics,
    compute_window_metrics,
    render_model_review_briefing_markdown,
    write_analysis_report,
)
from src.core import DatasetSnapshot

EXP_DIR = REPO_ROOT / 'research' / 'analysis_segment_effect_matrix_v1'
WINDOWS = {
    'early': ('2026-01-01', '2026-01-04'),
    'stress_window': ('2026-01-05', '2026-01-07'),
}


In [2]:
def daily_frame() -> pd.DataFrame:
    dates = list(pd.date_range('2026-01-01', periods=8, freq='D')) * 2
    regimes = ['bull', 'bull', 'bull', 'neutral', 'stress', 'stress', 'stress', 'downtrend'] * 2
    return pd.DataFrame(
        {
            'date': dates,
            'model': ['ridge_mse'] * 8 + ['gcn_mse'] * 8,
            'regime': regimes,
            'nav': [
                1.00, 1.02, 1.04, 1.03, 1.01, 0.99, 1.00, 1.01,
                1.00, 1.04, 1.08, 1.07, 1.02, 0.96, 0.97, 0.98,
            ],
            'benchmark_nav': [1.0, 1.01, 1.02, 1.02, 1.00, 0.99, 1.00, 1.00] * 2,
            'position': [0.2, 0.3, 0.5, 0.1, 0.4, 0.2, 0.1, 0.2] * 2,
            'turnover': [0.0, 0.1, 0.2, 0.4, 0.3, 0.1, 0.1, 0.2] * 2,
            'vix': [17, 18, 19, 24, 35, 32, 31, 22] * 2,
            'trend': [0.02, 0.03, 0.04, 0.00, -0.05, -0.06, -0.04, -0.02] * 2,
            'drawdown': [0.0, 0.0, 0.0, -0.01, -0.12, -0.14, -0.10, -0.04] * 2,
        }
    )

daily = daily_frame()
daily.head()


,date,model,regime,nav,benchmark_nav,position,turnover,vix,trend,drawdown
0,2026-01-01,ridge_mse,bull,1.00,1.00,0.2,0.0,17,0.02,0.00
1,2026-01-02,ridge_mse,bull,1.02,1.01,0.3,0.1,18,0.03,0.00
2,2026-01-03,ridge_mse,bull,1.04,1.02,0.5,0.2,19,0.04,0.00
3,2026-01-04,ridge_mse,neutral,1.03,1.02,0.1,0.4,24,0.00,-0.01
4,2026-01-05,ridge_mse,stress,1.01,1.00,0.4,0.3,35,-0.05,-0.12


In [3]:
analysis = analyze_model_effects_by_regime_and_window(
    daily,
    windows=WINDOWS,
    baseline_model='ridge_mse',
)
segment_effect_matrix = analysis.segment_effect_matrix
briefing = build_model_review_briefing(analysis, max_rows=8)

assert not segment_effect_matrix.empty
assert not briefing.empty
assert 'window_regime' in set(segment_effect_matrix['segment_type'])
assert {'weakness', 'strength'}.issubset(set(briefing['review_signal']))
assert analysis.summary['segment_effect_matrix_count'] == len(segment_effect_matrix)

segment_effect_matrix.loc[:, [
    'segment_type', 'segment', 'best_relative_return_model',
    'worst_relative_return_model', 'relative_return_pct_spread', 'review_note'
]]


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


,segment_type,segment,best_relative_return_model,worst_relative_return_model,relative_return_pct_spread,review_note
0,all,all,ridge_mse,gcn_mse,3.000000,ridge_mse 在 all 领先；gcn_mse 明显弱于基准，需检查该分段适配。
1,regime,bull,gcn_mse,ridge_mse,4.000000,gcn_mse 在 bull 强于基准 ridge_mse，可作为该 regime/wind...
2,regime,downtrend,gcn_mse,gcn_mse,0.000000,downtrend 模型差异较小，需结合回撤、换手和组合互补性判断。
3,regime,neutral,gcn_mse,gcn_mse,0.000000,neutral 模型差异较小，需结合回撤、换手和组合互补性判断。
4,regime,stress,ridge_mse,gcn_mse,3.911862,ridge_mse 在 stress 领先；gcn_mse 明显弱于基准，需检查该分段适配。
5,window,early,gcn_mse,ridge_mse,4.000000,gcn_mse 在 early 强于基准 ridge_mse，可作为该 regime/win...
6,window,stress_window,ridge_mse,gcn_mse,3.911862,ridge_mse 在 stress_window 领先；gcn_mse 明显弱于基准，需检...
7,window_regime,early::bull,gcn_mse,ridge_mse,4.000000,gcn_mse 在 early::bull 强于基准 ridge_mse，可作为该 regi...
8,window_regime,early::neutral,gcn_mse,gcn_mse,0.000000,early::neutral 模型差异较小，需结合回撤、换手和组合互补性判断。
9,window_regime,stress_window::stress,ridge_mse,gcn_mse,3.911862,ridge_mse 在 stress_window::stress 领先；gcn_mse 明...


In [4]:
report = build_regime_window_analysis_report(
    artifact_id='analysis_segment_effect_matrix_v1',
    effect_analysis=analysis,
    model_comparison=compare_models(daily),
    regime_metrics=compute_regime_metrics(daily),
    window_metrics=compute_window_metrics(daily, windows=WINDOWS),
    summary={
        'status': 'segment_effect_matrix_completed',
        'external_references': [
            'https://qlib.readthedocs.io/en/v0.8.3/reference/api.html#qlib.workflow.record_temp.PortAnaRecord',
            'https://qlib.readthedocs.io/en/v0.6.2/component/report.html',
            'https://github.com/microsoft/qlib/blob/main/docs/component/recorder.rst',
        ],
        'analysis_summary_zh': '新增 segment_effect_matrix 和 model_review_briefing，用于按 all/regime/window/window_regime 对比模型差异。',
    },
)
dataset = DatasetSnapshot(
    name='unit_soxl_analysis_segment_effect_matrix',
    universe=('SOXL', 'SOXX'),
    start='2026-01-01',
    end='2026-01-08',
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=('synthetic_unit_daily_frame',),
)
export = write_analysis_report(report, EXP_DIR, dataset=dataset)
(EXP_DIR / 'model_review_briefing.md').write_text(
    render_model_review_briefing_markdown(briefing, title='SOXL 模型差异 Review'),
    encoding='utf-8',
)
export


{'schema_version': 'analysis.report.export.v1',
 'artifact_id': 'analysis_segment_effect_matrix_v1',
 'artifact_type': 'analysis_report',
 'paths': {'summary': 'analysis_summary.json',
  'model_comparison': 'model_comparison.csv',
  'regime_metrics': 'regime_metrics.csv',
  'window_metrics': 'window_metrics.csv',
  'window_regime_metrics': 'window_regime_metrics.csv',
  'qlib_return_frame': 'qlib_return_frame.csv',
  'qlib_risk_metrics': 'qlib_risk_metrics.csv',
  'segment_metrics': 'segment_metrics.csv',
  'segment_effect_matrix': 'segment_effect_matrix.csv',
  'model_effect_deltas': 'model_effect_deltas.csv',
  'model_review_briefing': 'model_review_briefing.csv',
  'failure_modes': 'failure_modes.csv',
  'manifest': 'analysis_manifest.json'},
 'table_counts': {'failure_modes': 4,
  'model_comparison': 2,
  'model_effect_deltas': 20,
  'model_review_briefing': 5,
  'qlib_return_frame': 16,
  'qlib_risk_metrics': 40,
  'regime_metrics': 8,
  'segment_effect_matrix': 10,
  'segment_met

In [5]:
saved_matrix = pd.read_csv(EXP_DIR / 'segment_effect_matrix.csv')
saved_briefing = pd.read_csv(EXP_DIR / 'model_review_briefing.csv')
summary_path = EXP_DIR / 'analysis_summary.json'

assert summary_path.exists()
assert len(saved_matrix) == len(segment_effect_matrix)
assert len(saved_briefing) >= 1
assert 'best_relative_return_model' in saved_matrix.columns
assert 'review_signal' in saved_briefing.columns

{
    'segment_effect_matrix_rows': len(saved_matrix),
    'model_review_briefing_rows': len(saved_briefing),
    'top_review_signal': saved_briefing.iloc[0]['review_signal'],
    'top_review_segment': saved_briefing.iloc[0]['segment'],
    'export_paths': export['paths'],
}


{'segment_effect_matrix_rows': 10,
 'model_review_briefing_rows': 5,
 'top_review_signal': 'weakness',
 'top_review_segment': 'stress',
 'export_paths': {'summary': 'analysis_summary.json',
  'model_comparison': 'model_comparison.csv',
  'regime_metrics': 'regime_metrics.csv',
  'window_metrics': 'window_metrics.csv',
  'window_regime_metrics': 'window_regime_metrics.csv',
  'qlib_return_frame': 'qlib_return_frame.csv',
  'qlib_risk_metrics': 'qlib_risk_metrics.csv',
  'segment_metrics': 'segment_metrics.csv',
  'segment_effect_matrix': 'segment_effect_matrix.csv',
  'model_effect_deltas': 'model_effect_deltas.csv',
  'model_review_briefing': 'model_review_briefing.csv',
  'failure_modes': 'failure_modes.csv',
  'manifest': 'analysis_manifest.json'}}

## 中文结论

- 做了什么：在 `src/analysis` 中新增 `segment_effect_matrix` 和 `model_review_briefing`，把 qlib 风格的收益/风险分段分析转成 dashboard/notebook 可直接消费的模型差异矩阵。
- 核心指标：合成样例生成 all/regime/window/window_regime 多层分段，`segment_effect_matrix` 输出 best/worst/baseline spread，`model_review_briefing` 输出 weakness/strength review 条目。
- regime 差异：`gcn_mse` 在 bull 分段强于 `ridge_mse`，但在 stress/stress_window 分段弱于基准；该差异被矩阵和 briefing 同时捕捉。
- 是否晋升：晋升为 `src/analysis` 框架能力；不代表任何交易模型晋升到 production。
- 风险：当前是合成样例，真实 SOXL 模型需要接入 qlib recorder 或 canonical backtest export 后复跑。
- 下一步：把真实 QuantBench/backtest 产物接到该矩阵，并让 dashboard/schedule 优先展示最差 stress/window_regime review 条目。